# 06 — Interactive Plotly Dashboard

This notebook builds **interactive Plotly visualisations** for the Customer Journey
Drop-Off Analyzer. It loads all processed data, creates inline charts, and
verifies the companion **`dashboard_react.html`** file.

Sections covered:
1. Conversion Funnel
2. Exit Reasons Analysis
3. Device Conversion Rates
4. RFM Customer Segments
5. Monthly Session Trends
6. Hourly Drop-off Pattern
7. Dashboard HTML Export Verification

In [ ]:
# Cell 1 — Setup & Load Data
import pandas as pd
import numpy as np
import sys, os
sys.path.append("..")

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

from src.funnel_analyzer import build_funnel, device_funnel, hourly_drop_pattern, top_exit_reasons, STAGE_ORDER

# Dark theme matching dashboard_react.html
DARK_TEMPLATE = dict(
    paper_bgcolor="#07080a",
    plot_bgcolor="#111318",
    font=dict(color="#e0ddd8", family="DM Sans, sans-serif"),
    title_font=dict(size=16, color="#f1ede6"),
)
ORANGE = "#f97316"
PALETTE = ["#f97316", "#fb923c", "#fbbf24", "#22c55e", "#3b82f6", "#a855f7", "#ef4444"]

# Load raw sessions
df = pd.read_csv("../data/raw/sessions.csv")
print(f"Loaded {len(df):,} sessions  |  Columns: {list(df.columns)}")

# Load processed data
funnel_df = pd.read_csv("../data/processed/funnel_summary.csv")
rfm_df = pd.read_csv("../data/processed/rfm_scores.csv")
print(f"Funnel stages: {len(funnel_df)}  |  RFM customers: {len(rfm_df)}")

In [ ]:
# Cell 2 — Interactive Conversion Funnel
fig_funnel = go.Figure(go.Funnel(
    y=funnel_df["stage"],
    x=funnel_df["users"],
    textinfo="value+percent initial",
    marker=dict(
        color=PALETTE[:len(funnel_df)],
        line=dict(color="#1a1d22", width=1)
    ),
    connector=dict(line=dict(color="#333", width=1)),
))
fig_funnel.update_layout(
    title="Customer Journey Conversion Funnel",
    **DARK_TEMPLATE,
    margin=dict(l=180, r=40, t=60, b=40),
    height=450,
)
fig_funnel.show()

In [ ]:
# Cell 3 — Exit Reasons by Stage (Top Drop-off Points)
exit_stages = ["landing", "browse", "product_detail", "add_to_cart", "checkout", "payment"]

fig_exit = make_subplots(
    rows=2, cols=3,
    subplot_titles=[s.replace("_", " ").title() for s in exit_stages],
    vertical_spacing=0.18, horizontal_spacing=0.08
)

for idx, stage in enumerate(exit_stages):
    row, col = (idx // 3) + 1, (idx % 3) + 1
    reasons = top_exit_reasons(df, stage)
    if reasons.empty:
        continue
    top5 = reasons.head(5)
    fig_exit.add_trace(
        go.Bar(
            x=top5.values,
            y=top5.index,
            orientation="h",
            marker_color=PALETTE[idx % len(PALETTE)],
            name=stage.replace("_", " ").title(),
            showlegend=False,
        ),
        row=row, col=col
    )

fig_exit.update_layout(
    title="Exit Reasons by Funnel Stage",
    **DARK_TEMPLATE,
    height=550,
    margin=dict(l=40, r=40, t=80, b=40),
)
fig_exit.update_yaxes(autorange="reversed")
fig_exit.show()

In [ ]:
# Cell 4 — Conversion Rate by Device
dev = device_funnel(df).reset_index()
display(dev)

fig_device = go.Figure()
fig_device.add_trace(go.Bar(
    x=dev["device"],
    y=dev["conv_rate"],
    marker_color=[ORANGE, "#3b82f6", "#fb923c"],
    text=dev["conv_rate"].apply(lambda x: f"{x}%"),
    textposition="outside",
    textfont=dict(color="#9ca3af"),
))
fig_device.update_layout(
    title="Conversion Rate by Device",
    yaxis_title="Conversion Rate (%)",
    xaxis_title="Device",
    **DARK_TEMPLATE,
    height=400,
    showlegend=False,
)
fig_device.show()

In [ ]:
# Cell 5 — RFM Customer Segments
seg_counts = rfm_df["segment"].value_counts().reset_index()
seg_counts.columns = ["segment", "count"]

seg_colors = {
    "Champions": "#f97316",
    "Loyal Customers": "#fb923c",
    "Potential Loyalists": "#fbbf24",
    "At Risk": "#3b82f6",
    "Lost": "#4b5563",
}
colors = [seg_colors.get(s, "#888") for s in seg_counts["segment"]]

fig_rfm = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "bar"}]],
    subplot_titles=["Segment Distribution", "Customer Count by Segment"],
    column_widths=[0.45, 0.55],
)

fig_rfm.add_trace(
    go.Pie(
        labels=seg_counts["segment"],
        values=seg_counts["count"],
        marker=dict(colors=colors),
        hole=0.45,
        textinfo="percent+label",
        textfont=dict(size=10),
    ),
    row=1, col=1
)

fig_rfm.add_trace(
    go.Bar(
        x=seg_counts["segment"],
        y=seg_counts["count"],
        marker_color=colors,
        text=seg_counts["count"],
        textposition="outside",
        textfont=dict(color="#9ca3af"),
        showlegend=False,
    ),
    row=1, col=2
)

fig_rfm.update_layout(
    title="RFM Customer Segmentation",
    **DARK_TEMPLATE,
    height=420,
    showlegend=False,
)
fig_rfm.show()

print("\nSegment breakdown:")
display(seg_counts)

In [ ]:
# Cell 6 — Monthly Session Trends & Conversion Over Time
monthly = df.groupby("month").agg(
    sessions=("session_id", "count"),
    conversions=("converted", "sum"),
).reset_index()
monthly["conv_rate"] = (monthly["conversions"] / monthly["sessions"] * 100).round(2)

fig_monthly = make_subplots(specs=[[{"secondary_y": True}]])

fig_monthly.add_trace(
    go.Bar(
        x=monthly["month"], y=monthly["sessions"],
        name="Sessions", marker_color=ORANGE, opacity=0.7,
    ),
    secondary_y=False,
)
fig_monthly.add_trace(
    go.Scatter(
        x=monthly["month"], y=monthly["conv_rate"],
        name="Conv Rate %", mode="lines+markers",
        line=dict(color="#22c55e", width=3),
        marker=dict(size=8),
    ),
    secondary_y=True,
)

fig_monthly.update_layout(
    title="Monthly Sessions & Conversion Rate (Jan–Jun 2024)",
    **DARK_TEMPLATE,
    height=400,
    legend=dict(x=0.01, y=0.99),
)
fig_monthly.update_yaxes(title_text="Sessions", secondary_y=False)
fig_monthly.update_yaxes(title_text="Conversion Rate (%)", secondary_y=True)
fig_monthly.show()

In [ ]:
# Cell 7 — Hourly Drop-off Pattern
hourly = hourly_drop_pattern(df).reset_index()

fig_hourly = go.Figure()
fig_hourly.add_trace(go.Scatter(
    x=hourly["hour"], y=hourly["drop_rate"],
    mode="lines+markers",
    name="Drop Rate %",
    line=dict(color=ORANGE, width=2),
    marker=dict(size=6, color=ORANGE),
    fill="tozeroy",
    fillcolor="rgba(249,115,22,0.15)",
))

fig_hourly.update_layout(
    title="Drop-off Rate by Hour of Day",
    xaxis_title="Hour (0–23)",
    yaxis_title="Drop-off Rate (%)",
    **DARK_TEMPLATE,
    height=380,
    xaxis=dict(dtick=2),
)
fig_hourly.show()

In [ ]:
# Cell 8 — Dashboard HTML Verification
dashboard_path = os.path.abspath("../outputs/dashboard_react.html")
exists = os.path.exists(dashboard_path)
size_kb = os.path.getsize(dashboard_path) / 1024 if exists else 0

print("═" * 55)
print("  Dashboard React HTML — Status")
print("═" * 55)
print(f"  Path  : {dashboard_path}")
print(f"  Exists: {'✅ Yes' if exists else '❌ No'}")
print(f"  Size  : {size_kb:.1f} KB")
print("═" * 55)

if exists and size_kb > 1:
    print("\n✅ Dashboard is ready — open in browser:")
    print(f"   file:///{dashboard_path.replace(os.sep, '/')}")
else:
    print("\n⚠️  Dashboard file is missing or empty!")

---

## Summary

| # | Chart | Type | Key Insight |
|---|-------|------|-------------|
| 1 | Conversion Funnel | Funnel | 15K landing → ~744 confirmed (4.96% overall) |
| 2 | Exit Reasons | Grouped Bar | Slow load, unclear nav and forced login are top exit reasons |
| 3 | Device Conversion | Bar | Desktop has highest conversion rate |
| 4 | RFM Segments | Pie + Bar | Potential Loyalists is the largest segment |
| 5 | Monthly Trends | Bar + Line | Session volume and conversion tracked over 6 months |
| 6 | Hourly Drop-off | Area | Drop-off varies by time of day |

The full interactive React dashboard is at `../outputs/dashboard_react.html`.